# From an insurer PDF to structured data, with a Pydantic AI agent
**PDF → text → Pydantic model → AI agent fills it → structured output**

The example is Anjali Sharma's settlement letter from Arogya: no claim number, and her name is misspelt.
Set `GEMINI_API_KEY` before starting Jupyter (or paste the key when asked in step 3).

## 1. Read the PDF
This is the letter Arogya e-mailed the hospital: `arogya_letter.pdf`, in this folder. We pull out its text first. No AI yet.

In [1]:
from pypdf import PdfReader

text = "\n".join(page.extract_text() for page in PdfReader("arogya_letter.pdf").pages)
print(text)

AAROGYA PLUS TPA SERVICES PVT. LTD.Third Party Administrator | IRDAI Licence No. 018
B-14, Sector 18, Noida 201301 | Provider desk: 0120 455 7000
Ref: APT/SA/10/233200 Date: 2026-10-15
The Medical Superintendent
Nirmaya Multispeciality Hospital
Survey No. 42, Baner Road, Pune 411045
Sub: Settlement Advice
Dear Sir/Madam,
We have processed the claim of aanjali sharma (admitted on 2026-09-11). Bill amount Rs. 3,11,600/- ; approved
Rs. 2,33,200/-. Disallowance: Investigation charges disallowed.
The amount will be remitted to your registered bank account within 7 working days. For any clarification please
contact our provider desk, quoting the reference above.
For Arogya Plus TPA Services Pvt. Ltd.
Authorised Signatory
Claims Team
This is a computer-generated letter and does not require a physical signature.


## 2. The Pydantic model: the shape we want
The same fields for every insurer, whatever their letter looks like. The `description`s tell the AI what each field means.

In [2]:
from typing import Literal
from pydantic import BaseModel, Field

class Settlement(BaseModel):
    insurer: str = Field(description="The insurance company or TPA that wrote the letter")
    claim_no: str | None = Field(description="The HOSPITAL's claim number, like HSP-1011, if the letter shows one, "
                                             "else null. Not the insurer's own reference number.")
    patient_name: str = Field(description="The patient's name exactly as written, typos included")
    billed: float = Field(description="Amount the hospital billed, in rupees, as a plain number")
    paid: float = Field(description="Amount the insurer approved / paid, in rupees, as a plain number")
    reason: str = Field(description="The insurer's reason for paying less, word for word; '' if paid in full")
    category: Literal["ROOM_LIMIT", "NOT_COVERED_ITEMS", "PATIENT_SHARE", "OTHER"] = Field(
        description="ROOM_LIMIT: room above the policy limit. NOT_COVERED_ITEMS: only consumables like gloves, masks. "
                    "PATIENT_SHARE: co-pay. OTHER: anything else (tests, implants, treatment) or nothing cut.")

print(list(Settlement.model_fields))

['insurer', 'claim_no', 'patient_name', 'billed', 'paid', 'reason', 'category']


## 3. The AI agent (Pydantic AI)
An agent = a model (Gemini) + instructions + the **output type**. Because the output type is `Settlement`, the agent must answer with a valid `Settlement`. This is how ClaimTrail's reader agent works (`integration/claimtrail/ai.py`).

In [3]:
import os
from getpass import getpass
from pydantic_ai import Agent, NativeOutput

os.environ["PYDANTIC_AI_NO_BANNER"] = "1"
os.environ.setdefault("GOOGLE_API_KEY", os.environ.pop("GEMINI_API_KEY", "") or getpass("Gemini API key: "))

agent = Agent(
    "google:gemini-3.1-flash-lite",
    output_type=NativeOutput(Settlement),   # Gemini's native JSON output, the same as the project's reader
    system_prompt="You read insurance settlement letters sent to a hospital and extract the settlement details.",
)
agent

Agent(model=GoogleModel(model_name='gemini-3.1-flash-lite'), name=None, end_strategy='graceful', model_settings=None, output_type=NativeOutput(outputs=<class '__main__.Settlement'>, name=None, description=None, strict=None, template=None))

## 4. The agent reads the letter and fills in the details
The result is a real `Settlement` object, already checked by Pydantic (numbers are numbers, the category is one of the four).

In [4]:
result = await agent.run(text)
settlement = result.output
settlement

Settlement(insurer='AAROGYA PLUS TPA SERVICES PVT. LTD.', claim_no=None, patient_name='aanjali sharma', billed=311600.0, paid=233200.0, reason='Investigation charges disallowed.', category='OTHER')

## 5. The structured details
Clean data, ready for the hospital system. The money decision is plain code, not AI.

In [5]:
import pandas as pd

print(settlement.model_dump_json(indent=2))

cut = settlement.billed - settlement.paid
decision = "Fair cut" if settlement.category in {"ROOM_LIMIT", "NOT_COVERED_ITEMS", "PATIENT_SHARE"} else "Unfair cut"
pd.DataFrame([{**settlement.model_dump(), "cut": cut, "decision": decision if cut > 100 else "Paid in full"}])

{
  "insurer": "AAROGYA PLUS TPA SERVICES PVT. LTD.",
  "claim_no": null,
  "patient_name": "aanjali sharma",
  "billed": 311600.0,
  "paid": 233200.0,
  "reason": "Investigation charges disallowed.",
  "category": "OTHER"
}


,insurer,claim_no,patient_name,billed,paid,reason,category,cut,decision
0,AAROGYA PLUS TPA SERVICES PVT. LTD.,None,aanjali sharma,311600.0,233200.0,Investigation charges disallowed.,OTHER,78400.0,Unfair cut
